# Tech Addiction Prediction: TabNet
In this notebook, we build a `TabNet` model (Attentive Interpretable Tabular Learning). 
TabNet is a neural network architecture designed specifically for tabular data. It brings the power of deep learning to tabular datasets without needing extensive feature engineering, and provides interpretability by selecting which features to reason from at each step.


In [ ]:
!pip install -q pytorch-tabnet


In [ ]:
import pandas as pd
import numpy as np
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)


## 1. Data Loading


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values


## 2. V5 Feature Engineering & Scaling
Like any Neural Network, TabNet is sensitive to feature scaling. We apply `StandardScaler` to numeric features and `LabelEncoder` or `OneHotEncoder` to categorical ones.


In [ ]:
def engineer_features(train, test):
    train = train.copy()
    test = test.copy()
    
    for df in [train, test]:
        # Original V2 Features
        df['weekend_delta'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
        df['social_media_prop'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['unaccounted_screen_time'] = df['daily_screen_time_hours'] - (df['social_media_hours'] + df['gaming_hours'] + df['work_study_hours'])
        
        # NEW V5 Features
        df['productivity_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['entertainment_ratio'] = (df['social_media_hours'] + df['gaming_hours']) / (df['daily_screen_time_hours'] + 1e-5)
        df['screen_to_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-5)
        df['interaction_intensity'] = df['notifications_per_day'] * df['app_opens_per_day']
        df['sleep_deprived'] = (df['sleep_hours'] < 6.5).astype(int)
        df['age_group'] = pd.cut(df['age'], bins=[0, 20, 30, 40, 50, 100], labels=False)
        
    return train, test

print("Engineering Deep V5 features...")
X, test_df_eng = engineer_features(X, test_df.drop(['id'], axis=1))

categorical_features = ['gender', 'academic_work_impact', 'stress_level', 'age_group']
numeric_features = [col for col in X.columns if col not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(test_df_eng)
print(f"Processed Train shape: {X_processed.shape}")


## 3. Stratified 5-Fold Cross Validation for TabNet


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation for TabNet...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds_proba = np.zeros(len(X_test_processed))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_train, X_val = X_processed[train_idx], X_processed[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    clf = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5,
        gamma=1.5, n_independent=2, n_shared=2,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        mask_type="entmax",
        scheduler_params=dict(mode="min", patience=10, min_lr=1e-5, factor=0.5),
        scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
        seed=42, verbose=1
    )
    
    clf.fit(
        X_train=X_train, y_train=y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        eval_name=['train', 'val'],
        eval_metric=['auc'],
        max_epochs=100 , patience=20,
        batch_size=2048, virtual_batch_size=256,
        num_workers=0,
        drop_last=False
    )
    
    # Predict OOF
    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]
    
    # Predict Test
    test_preds_proba += clf.predict_proba(X_test_processed)[:, 1] / 5

print("\nCross-Validation complete!")


### Metric Evaluation


In [ ]:
print("Evaluating OOF predictions...")
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("TabNet (5-Fold OOF) Performance:")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)


## 4. Submission


In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
